# 🏏 SmartCricket360 – IPL Data Engineering Framework

## Dataset Overview
This notebook processes the complete IPL dataset using the **Medallion Architecture**:

* **Bronze Layer** - Raw data ingestion (both delivery and match info files)
* **Silver Layer** - Cleaned, standardized, and enriched data
* **Gold Layer** - Business-ready aggregations with standard IPL KPIs

## Data Structure
* **Delivery Files** (`{match_id}.csv`) - Ball-by-ball records
* **Match Info Files** (`{match_id}_info.csv`) - Match metadata (teams, venue, dates, etc.)

## Standard IPL KPIs
* Batting: Total runs, strike rate, fours, sixes, boundaries, batting average, centuries, fifties
* Bowling: Wickets, economy rate, bowling average
* Team: Win/loss records, NRR, home/away performance
* Match: Toss impact, venue statistics, season trends

## Step 1: Explore Data Structure

First, let's examine the schema and sample data from both file types to understand what we're working with.

In [0]:
# Read a sample delivery file to see the schema
delivery_sample = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/ipl_complete/ipl_complete_analytics/ipl_complete_dataset/ipl_complete_data_csv/335982.csv")

print("=== DELIVERY FILE SCHEMA ===")
delivery_sample.printSchema()

print("\n=== SAMPLE DELIVERY DATA (5 rows) ===")
display(delivery_sample.limit(5))

=== DELIVERY FILE SCHEMA ===
root
 |-- match_id: integer (nullable = true)
 |-- season: date (nullable = true)
 |-- start_date: date (nullable = true)
 |-- venue: string (nullable = true)
 |-- innings: integer (nullable = true)
 |-- ball: double (nullable = true)
 |-- batting_team: string (nullable = true)
 |-- bowling_team: string (nullable = true)
 |-- striker: string (nullable = true)
 |-- non_striker: string (nullable = true)
 |-- bowler: string (nullable = true)
 |-- runs_off_bat: integer (nullable = true)
 |-- extras: integer (nullable = true)
 |-- wides: integer (nullable = true)
 |-- noballs: string (nullable = true)
 |-- byes: integer (nullable = true)
 |-- legbyes: integer (nullable = true)
 |-- penalty: string (nullable = true)
 |-- wicket_type: string (nullable = true)
 |-- player_dismissed: string (nullable = true)
 |-- other_wicket_type: string (nullable = true)
 |-- other_player_dismissed: string (nullable = true)


=== SAMPLE DELIVERY DATA (5 rows) ===


match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,bowler,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed
335982,2007-08-01,2008-04-18,M Chinnaswamy Stadium,1,0.1,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,P Kumar,0,1,null,null,null,1,null,null,null,null,null
335982,2007-08-01,2008-04-18,M Chinnaswamy Stadium,1,0.2,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,null,null,null,null,null,null,null,null,null
335982,2007-08-01,2008-04-18,M Chinnaswamy Stadium,1,0.3,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,1,1,null,null,null,null,null,null,null,null
335982,2007-08-01,2008-04-18,M Chinnaswamy Stadium,1,0.4,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,null,null,null,null,null,null,null,null,null
335982,2007-08-01,2008-04-18,M Chinnaswamy Stadium,1,0.5,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,0,0,null,null,null,null,null,null,null,null,null


In [0]:
# Read a sample match info file to see the schema
match_info_sample = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/ipl_complete/ipl_complete_analytics/ipl_complete_dataset/ipl_complete_data_csv/335982_info.csv")

print("=== MATCH INFO FILE SCHEMA ===")
match_info_sample.printSchema()

print("\n=== SAMPLE MATCH INFO DATA ===")
display(match_info_sample)

=== MATCH INFO FILE SCHEMA ===
root
 |-- version: string (nullable = true)
 |-- 2.1.0: string (nullable = true)


=== SAMPLE MATCH INFO DATA ===


version,2.1.0
info,balls_per_over
info,team
info,team
info,gender
info,season
info,date
info,event
info,match_number
info,venue
info,city


## 🔥 Step 2: Bronze Layer - Raw Data Ingestion

Load all CSV files as-is into Delta Lake format for ACID compliance and time travel capability.

In [0]:
%sql
-- Create Unity Catalog volumes for Medallion architecture layers

-- Create Bronze volume
CREATE VOLUME IF NOT EXISTS ipl_complete.ipl_complete_analytics.bronze
COMMENT 'Bronze layer - raw IPL data ingestion';

-- Create Silver volume  
CREATE VOLUME IF NOT EXISTS ipl_complete.ipl_complete_analytics.silver
COMMENT 'Silver layer - cleaned and standardized IPL data';

-- Create Gold volume
CREATE VOLUME IF NOT EXISTS ipl_complete.ipl_complete_analytics.gold
COMMENT 'Gold layer - business-ready IPL analytics and KPIs';

SHOW VOLUMES IN ipl_complete.ipl_complete_analytics;

database,volume_name
ipl_complete_analytics,bronze
ipl_complete_analytics,gold
ipl_complete_analytics,ipl_complete_dataset
ipl_complete_analytics,silver


In [0]:
from pyspark.sql.functions import col

# Define paths
base_path = "/Volumes/ipl_complete/ipl_complete_analytics/ipl_complete_dataset/ipl_complete_data_csv"
bronze_path = "/Volumes/ipl_complete/ipl_complete_analytics/bronze"

# Read all delivery files (excludes _info files)
deliveries_bronze_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{base_path}/[0-9]*.csv")

print(f"Raw records loaded: {deliveries_bronze_raw.count():,}")

# Filter out metadata rows - keep only actual delivery records
# Metadata rows have match_id as string like 'info', 'version', etc.
# Filter by checking if match_id and season are valid
# Season can be: numeric (2009, 2017) OR slash format (2007/08, 2009/10, 2020/21)
deliveries_bronze = deliveries_bronze_raw.filter(
    col("match_id").rlike("^[0-9]+$") & 
    col("season").rlike("^[0-9]{4}(/[0-9]{2})?$")
)

# Cast penalty column to string to avoid schema merge conflicts
if "penalty" in deliveries_bronze.columns:
    deliveries_bronze = deliveries_bronze.withColumn("penalty", col("penalty").cast("string"))

# Transform season to full 4-digit year
# For slash format: extract ending year (2007/08 → 2008, 2009/10 → 2010)
# Exception: 2020/21 → 2020 (to avoid conflict with separate 2021 season)
# For numeric format (2009), keep as is
from pyspark.sql.functions import when, substring, concat

deliveries_bronze = deliveries_bronze.withColumn(
    "season",
    when(
        col("season") == "2020/21",
        "2020"  # Special case: map to 2020 to avoid conflict with 2021
    ).when(
        col("season").contains("/"),
        # For "2007/08": take "20" + "08" = "2008"
        concat(
            substring(col("season"), 1, 2),  # Century: "20"
            substring(col("season"), 6, 2)   # Year after slash: "08"
        )
    ).otherwise(col("season"))
)

print(f"Actual delivery records after filtering: {deliveries_bronze.count():,}")
print(f"Columns: {len(deliveries_bronze.columns)}")
print(f"Seasons: {sorted([row.season for row in deliveries_bronze.select('season').distinct().collect()])}")

# Save to Bronze layer
deliveries_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{bronze_path}/deliveries")
print("\n✓ Deliveries saved to Bronze layer")

print("\n=== BRONZE DATA PREVIEW (10 rows) ===")
display(deliveries_bronze.select("match_id", "season", "striker", "bowler", "runs_off_bat", "extras", "wicket_type").limit(10))

Raw records loaded: 382,699
Actual delivery records after filtering: 295,732
Columns: 22
Seasons: ['2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']

✓ Deliveries saved to Bronze layer

=== BRONZE DATA PREVIEW (10 rows) ===


match_id,season,striker,bowler,runs_off_bat,extras,wicket_type
1359517,2023,V Kohli,KH Pandya,4,0,null
1359517,2023,V Kohli,KH Pandya,0,0,null
1359517,2023,V Kohli,KH Pandya,0,0,null
1359517,2023,V Kohli,KH Pandya,0,0,null
1359517,2023,V Kohli,KH Pandya,1,0,null
1359517,2023,F du Plessis,KH Pandya,0,0,null
1359517,2023,V Kohli,MP Stoinis,1,0,null
1359517,2023,F du Plessis,MP Stoinis,2,0,null
1359517,2023,F du Plessis,MP Stoinis,2,0,null
1359517,2023,F du Plessis,MP Stoinis,1,0,null


In [0]:
# Read all match info files
match_info_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .load(f"{base_path}/*_info.csv")

print(f"Total match info records: {match_info_bronze.count():,}")
print(f"\nSample structure:")
match_info_bronze.show(10, truncate=False)

# Save to Bronze layer
match_info_bronze.write.format("delta").mode("overwrite").save(f"{bronze_path}/match_info_raw")
print("✓ Match info saved to Bronze layer")

print("\n=== MATCH INFO PREVIEW (10 rows) ===")
display(match_info_bronze.limit(10))

Total match info records: 86,967

Sample structure:
+-------+--------------+
|version|2.2.0         |
+-------+--------------+
|info   |balls_per_over|
|info   |team          |
|info   |team          |
|info   |gender        |
|info   |season        |
|info   |date          |
|info   |event         |
|info   |match_number  |
|info   |venue         |
|info   |city          |
+-------+--------------+
only showing top 10 rows
✓ Match info saved to Bronze layer

=== MATCH INFO PREVIEW (10 rows) ===


version,2.2.0
info,balls_per_over
info,team
info,team
info,gender
info,season
info,date
info,event
info,match_number
info,venue
info,city


## ✨ Step 3: Silver Layer - Transform & Enrich

Transform Bronze data into clean, standardized format:
* Add derived columns (boundaries, wickets, legal balls)
* Calculate total runs (runs_off_bat + extras)
* Handle null values
* Partition by season for query performance

In [0]:
from pyspark.sql.functions import col, when, sum as spark_sum, count, coalesce, lit

# Define paths
bronze_path = "/Volumes/ipl_complete/ipl_complete_analytics/bronze"
silver_path = "/Volumes/ipl_complete/ipl_complete_analytics/silver"

# Load Bronze deliveries
deliveries_bronze = spark.read.format("delta").load(f"{bronze_path}/deliveries")

print(f"Bronze records loaded: {deliveries_bronze.count():,}")

# Transform and enrich for Silver layer
deliveries_silver = deliveries_bronze \
    .withColumn("total_runs", col("runs_off_bat") + coalesce(col("extras"), lit(0))) \
    .withColumn("is_four", when(col("runs_off_bat") == 4, 1).otherwise(0)) \
    .withColumn("is_six", when(col("runs_off_bat") == 6, 1).otherwise(0)) \
    .withColumn("is_boundary", when((col("runs_off_bat") == 4) | (col("runs_off_bat") == 6), 1).otherwise(0)) \
    .withColumn("is_wicket", when(col("wicket_type").isNotNull(), 1).otherwise(0)) \
    .withColumn("legal_ball", when((coalesce(col("wides"), lit(0)) == 0) & (coalesce(col("noballs"), lit(0)) == 0), 1).otherwise(0))

print(f"Silver transformations applied: {deliveries_silver.count():,} records")
print(f"Seasons in Silver: {sorted([row.season for row in deliveries_silver.select('season').distinct().collect()])}")

# Save to Silver layer (partitioned by season for performance)
deliveries_silver.write.format("delta").mode("overwrite").partitionBy("season").save(f"{silver_path}/deliveries")

print("\n✓ Silver layer saved successfully")
print("\n=== SILVER DATA PREVIEW (10 rows) ===")
display(deliveries_silver.select("match_id", "season", "striker", "bowler", "runs_off_bat", "total_runs", "is_boundary", "is_wicket").limit(10))

Bronze records loaded: 295,732
Silver transformations applied: 295,732 records
Seasons in Silver: ['2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']

✓ Silver layer saved successfully

=== SILVER DATA PREVIEW (10 rows) ===


match_id,season,striker,bowler,runs_off_bat,total_runs,is_boundary,is_wicket
1082591,2017,DA Warner,TS Mills,0,0,0,0
1082591,2017,DA Warner,TS Mills,0,0,0,0
1082591,2017,DA Warner,TS Mills,4,4,1,0
1082591,2017,DA Warner,TS Mills,0,0,0,0
1082591,2017,DA Warner,TS Mills,0,2,0,0
1082591,2017,S Dhawan,TS Mills,0,0,0,0
1082591,2017,S Dhawan,TS Mills,0,1,0,0
1082591,2017,S Dhawan,A Choudhary,1,1,0,0
1082591,2017,DA Warner,A Choudhary,4,4,1,0
1082591,2017,DA Warner,A Choudhary,0,1,0,0


In [0]:
# Preview Silver data before aggregation to verify data quality
print("=== SILVER DATA PREVIEW ===")
print(f"Total records: {deliveries_silver.count():,}")
print(f"\nDistinct seasons: {sorted([row.season for row in deliveries_silver.select('season').distinct().collect()])}")
print(f"Distinct match_ids: {deliveries_silver.select('match_id').distinct().count()}")
print(f"\nSample delivery records:")
display(deliveries_silver.select("match_id", "season", "striker", "bowler", "runs_off_bat", "is_boundary", "is_wicket").limit(10))

=== SILVER DATA PREVIEW ===
Total records: 295,732

Distinct seasons: ['2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
Distinct match_ids: 1243

Sample delivery records:


match_id,season,striker,bowler,runs_off_bat,is_boundary,is_wicket
1082591,2017,DA Warner,TS Mills,0,0,0
1082591,2017,DA Warner,TS Mills,0,0,0
1082591,2017,DA Warner,TS Mills,4,1,0
1082591,2017,DA Warner,TS Mills,0,0,0
1082591,2017,DA Warner,TS Mills,0,0,0
1082591,2017,S Dhawan,TS Mills,0,0,0
1082591,2017,S Dhawan,TS Mills,0,0,0
1082591,2017,S Dhawan,A Choudhary,1,0,0
1082591,2017,DA Warner,A Choudhary,4,1,0
1082591,2017,DA Warner,A Choudhary,0,0,0


## 🏆 Step 4: Gold Layer - Lakeflow Pipeline

The Gold layer is now handled by a **Lakeflow Spark Declarative Pipeline** that automatically creates business-ready analytics tables with:

* **5 Gold Tables**: batting_stats, bowling_stats, team_performance, match_summary, venue_stats
* **12 Data Quality Expectations**: Automated validation of null values, ranges, and business rules
* **Automatic Dependency Resolution**: Pipeline manages table refresh order
* **Built-in Monitoring**: Event logs, lineage visualization, and performance metrics

The pipeline reads from the Silver layer and publishes managed tables to Unity Catalog at `ipl_complete.ipl_complete_analytics`.

In [0]:
import time
from databricks.sdk import WorkspaceClient

print("=== TRIGGERING GOLD LAYER LAKEFLOW PIPELINE ===\n")

# Initialize workspace client
w = WorkspaceClient()

# Pipeline configuration
pipeline_id = "8e58d57e-aed6-422e-b64f-7abe6c820963"
pipeline_name = "IPL Gold Layer Pipeline"

try:
    # Start pipeline update
    print(f"Starting {pipeline_name}...")
    update = w.pipelines.start_update(
        pipeline_id=pipeline_id,
        full_refresh=False
    )
    
    update_id = update.update_id
    print(f"✓ Pipeline update started (Update ID: {update_id})\n")
    
    # Poll for completion
    print("Waiting for pipeline to complete...")
    status = "RUNNING"
    
    while status in ["INITIALIZING", "RUNNING", "WAITING_FOR_RESOURCES"]:
        time.sleep(10)  # Check every 10 seconds
        
        # Get current status
        update_info = w.pipelines.get_update(pipeline_id=pipeline_id, update_id=update_id)
        status = update_info.update.state.value
        
        print(f"  Status: {status}")
    
    # Final status
    print(f"\n{'='*60}")
    if status == "COMPLETED":
        print("✅ PIPELINE COMPLETED SUCCESSFULLY")
        
        # Get dataset details
        print(f"\n{'='*60}")
        print("GOLD LAYER TABLES CREATED/UPDATED:")
        print(f"{'='*60}\n")
        
        # Query row counts from Unity Catalog
        tables = [
            "batting_stats",
            "bowling_stats", 
            "team_performance",
            "match_summary",
            "venue_stats"
        ]
        
        for table in tables:
            try:
                count_df = spark.sql(f"SELECT COUNT(*) as count FROM ipl_complete.ipl_complete_analytics.{table}")
                row_count = count_df.collect()[0]["count"]
                print(f"  ✓ {table}: {row_count:,} rows")
            except Exception as e:
                print(f"  ⚠ {table}: Unable to get row count")
        
        print(f"\n{'='*60}")
        print("🎉 Gold layer is now up-to-date!")
        print("Your Bronze → Silver → Gold pipeline is complete.\n")
        
    elif status == "FAILED":
        print("❌ PIPELINE FAILED")
        print(f"\nCheck pipeline logs for details:")
        print(f"Pipeline ID: {pipeline_id}")
        print(f"Update ID: {update_id}\n")
        
    elif status == "CANCELED":
        print("⚠️ PIPELINE CANCELED")
        
except Exception as e:
    print(f"\n❌ ERROR: {str(e)}")
    print(f"\nFailed to trigger pipeline. Please check:")
    print(f"  1. Pipeline ID is correct: {pipeline_id}")
    print(f"  2. You have permissions to run the pipeline")
    print(f"  3. Pipeline configuration is valid\n")
    raise

=== TRIGGERING GOLD LAYER LAKEFLOW PIPELINE ===

Starting IPL Gold Layer Pipeline...
✓ Pipeline update started (Update ID: a063b88d-52be-4cd1-8e34-a067b4a3ed22)

Waiting for pipeline to complete...
  Status: WAITING_FOR_RESOURCES
  Status: INITIALIZING
  Status: INITIALIZING
  Status: RUNNING
  Status: RUNNING
  Status: RUNNING
  Status: COMPLETED

✅ PIPELINE COMPLETED SUCCESSFULLY

GOLD LAYER TABLES CREATED/UPDATED:

  ✓ batting_stats: 2,959 rows
  ✓ bowling_stats: 2,203 rows
  ✓ team_performance: 166 rows
  ✓ match_summary: 1,243 rows
  ✓ venue_stats: 60 rows

🎉 Gold layer is now up-to-date!
Your Bronze → Silver → Gold pipeline is complete.



### 🔍 What the Lakeflow Pipeline Does

The **IPL Gold Layer Pipeline** is a Lakeflow Spark Declarative Pipeline that transforms Silver data into business-ready analytics:

#### Gold Tables Created
1. **batting_stats** - Player batting performance by season (runs, strike rate, boundaries, averages)
2. **bowling_stats** - Bowler performance by season (wickets, economy rate, bowling average)
3. **team_performance** - Team statistics by season (wins, losses, total runs, NRR)
4. **match_summary** - Match-level aggregations (total runs, wickets, boundaries per match)
5. **venue_stats** - Venue-specific insights (matches played, average scores by venue)

#### Data Quality Expectations
The pipeline includes **12 automated expectations** to ensure data quality:
* Null checks on key columns (striker, bowler, team, match_id, venue)
* Range validations (runs ≥ 0, strike rate ≥ 0, economy rate ≥ 0)
* Business rule enforcement (wickets ≤ 10 per match, valid season ranges)

#### Benefits of Lakeflow
* ✅ **Automatic dependency management** - No manual ordering of transformations
* ✅ **Built-in monitoring** - Event logs track every pipeline run
* ✅ **Lineage visualization** - See data flow from Bronze → Silver → Gold
* ✅ **Data quality tracking** - Expectations report violations automatically
* ✅ **Incremental updates** - Only processes new data on subsequent runs

#### Hybrid Approach
This notebook uses a **hybrid architecture**:
* **Bronze/Silver** - Run in notebook for fast one-time CSV loads
* **Gold** - Run in Lakeflow pipeline for production-grade quality and monitoring

## 📊 Step 5: Create Unity Catalog Views

Create permanent UC views over the Gold layer Delta tables for easy querying from:
* Databricks SQL dashboards
* BI tools (Power BI, Tableau)
* Other notebooks and queries

In [0]:
%sql
-- Set catalog context
USE CATALOG ipl_complete;

-- Create Unity Catalog views pointing to Gold Delta tables

-- Batting statistics view
CREATE OR REPLACE VIEW ipl_complete.ipl_complete_analytics.batting_stats_vw AS
SELECT * FROM delta.`/Volumes/ipl_complete/ipl_complete_analytics/gold/batting_stats`;

-- Bowling statistics view
CREATE OR REPLACE VIEW ipl_complete.ipl_complete_analytics.bowling_stats_vw AS
SELECT * FROM delta.`/Volumes/ipl_complete/ipl_complete_analytics/gold/bowling_stats`;

-- Team performance view
CREATE OR REPLACE VIEW ipl_complete.ipl_complete_analytics.team_performance_vw AS
SELECT * FROM delta.`/Volumes/ipl_complete/ipl_complete_analytics/gold/team_performance`;

-- Match summary view
CREATE OR REPLACE VIEW ipl_complete.ipl_complete_analytics.match_summary_vw AS
SELECT * FROM delta.`/Volumes/ipl_complete/ipl_complete_analytics/gold/match_summary`;

-- Venue statistics view
CREATE OR REPLACE VIEW ipl_complete.ipl_complete_analytics.venue_stats_vw AS
SELECT * FROM delta.`/Volumes/ipl_complete/ipl_complete_analytics/gold/venue_stats`;

SHOW VIEWS IN ipl_complete.ipl_complete_analytics;

namespace,viewName,isTemporary,isMaterialized,isMetric
ipl_complete_analytics,batting_stats,false,true,false
ipl_complete_analytics,batting_stats_vw,false,false,false
ipl_complete_analytics,bowling_stats,false,true,false
ipl_complete_analytics,bowling_stats_vw,false,false,false
ipl_complete_analytics,match_summary,false,true,false
ipl_complete_analytics,match_summary_vw,false,false,false
ipl_complete_analytics,team_performance,false,true,false
ipl_complete_analytics,team_performance_vw,false,false,false
ipl_complete_analytics,venue_stats,false,true,false
ipl_complete_analytics,venue_stats_vw,false,false,false


## ✅ Pipeline Complete!

### Hybrid Medallion Architecture

This pipeline uses a **hybrid approach** combining notebook flexibility with Lakeflow production features:

#### Bronze Layer (Notebook)
* ✓ Delivery files: Ball-by-ball records (295,732 records)
* ✓ Match info files: Match metadata
* ✓ Season transformation: Handles both numeric (2009) and slash formats (2007/08 → 2008)
* ✓ Schema conflict resolution: penalty column cast to string

#### Silver Layer (Notebook)
* ✓ Enriched deliveries with derived metrics (total_runs, boundaries, wickets, legal balls)
* ✓ Null handling with coalesce patterns
* ✓ Partitioned by season for query performance
* ✓ All historical IPL seasons recovered (2008-2026)

#### Gold Layer (Lakeflow Pipeline)
* ✓ **5 managed tables** via Lakeflow Spark Declarative Pipeline
* ✓ **12 data quality expectations** with automated validation
* ✓ **Automatic dependency resolution** - no manual ordering required
* ✓ **Event logs & lineage** - built-in monitoring and observability
* ✓ **Incremental updates** - only processes changed data

#### Unity Catalog Integration
* ✓ All Gold tables published to `ipl_complete.ipl_complete_analytics`
* ✓ SQL-queryable views for dashboards and BI tools
* ✓ Ready for Databricks SQL, Power BI, Tableau

### Why This Hybrid Approach?

**Notebook (Bronze/Silver)**:
* Fast one-time CSV loads with custom transformations
* Easy debugging and iteration
* Full control over complex season logic

**Lakeflow (Gold)**:
* Production-grade data quality monitoring
* Automatic retry and error handling
* Built-in lineage and observability
* Incremental processing for ongoing updates

### Next Steps
1. **Query Gold tables** using Databricks SQL or notebooks
2. **Create dashboards** with batting/bowling/team analytics
3. **Set up refresh schedule** - Run "All" on this notebook to update Bronze → Silver → Gold
4. **Monitor pipeline** - Check [IPL Gold Layer Pipeline](#pipeline-monitoring-8e58d57e-aed6-422e-b64f-7abe6c820963) for data quality metrics
5. **Extend Gold layer** - Add more KPIs by editing the pipeline's `gold_layer.py` file